# PHASE 11 - Save the Best Model

The ablation study selected the time + weather + lag feature set using validation RMSE. This notebook trains that selected configuration on train plus validation data and saves it for later prediction.

The test set remains untouched until the final evaluation in this notebook.

In [ ]:
# Nhập các thư viện cần thiết
from pathlib import Path  # Làm việc với đường dẫn file
from time import perf_counter  # Đo thời gian thực thi
import pickle  # Lưu và tải các object Python

import pandas as pd  # Xử lý dữ liệu
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Các hàm tính metrics
from xgboost import XGBRegressor  # Model XGBoost

# Hàm tìm thư mục gốc của project
def find_project_root():
    # Kiểm tra từng vị trí xem có file train.csv không
    for candidate in [Path("."), Path("..")]:
        if (candidate / "data/processed/train.csv").exists():
            return candidate
    # Nếu không tìm thấy, báo lỗi
    raise FileNotFoundError("Run PHASE 3 first.")

# Tìm thư mục gốc
project_root = find_project_root()

# Định nghĩa các đường dẫn
processed_dir = project_root / "data/processed"  # Thư mục dữ liệu
metrics_dir = project_root / "results/metrics"  # Thư mục metrics
models_dir = project_root / "models"  # Thư mục mô hình

# Tải dữ liệu train, validation, test
train_df = pd.read_csv(processed_dir / "train.csv", parse_dates=["timestamp", "dteday"])
validation_df = pd.read_csv(processed_dir / "validation.csv", parse_dates=["timestamp", "dteday"])
test_df = pd.read_csv(processed_dir / "test.csv", parse_dates=["timestamp", "dteday"])

# Tải kết quả ablation study
ablation_df = pd.read_csv(metrics_dir / "ablation_study_metrics.csv")

# Tải kết quả tìm kiếm siêu tham số từ PHASE 6
search_df = pd.read_csv(metrics_dir / "xgboost_validation_search.csv")

# === LẤY KẾT QUẢ TỐT NHẤT TỪ ABLATION STUDY ===
# Lấy kết quả tốt nhất trên validation (sắp xếp theo RMSE)
best_validation = ablation_df[ablation_df["split"] == "validation"].sort_values("RMSE").iloc[0]

# Kiểm tra rằng kết quả tốt nhất là feature set C (time + weather + lag)
assert best_validation["experiment"] == "C_time_weather_lag"

# === ĐỊNH NGHĨA FEATURES ===
# Nhóm 1: Features liên quan đến thời gian
time_features = ["yr", "mnth", "hr", "holiday", "weekday", "workingday", "season", "hour", "day", "month", "year", "day_of_week", "day_of_year", "is_weekend", "is_workingday", "rush_hour"]

# Nhóm 2: Features liên quan đến thời tiết
weather_features = ["weathersit", "temp", "atemp", "hum", "windspeed"]

# Nhóm 3: Lag features (giá trị trong quá khứ)
lag_features = ["lag_1", "lag_2", "lag_24", "lag_168"]

# Ghép 3 nhóm features đã chọn
feature_columns = time_features + weather_features + lag_features

# Cột mục tiêu
target_column = "cnt"

# === GHÉP TRAIN VÀ VALIDATION ===
# Ghép train và validation để huấn luyện mô hình cuối cùng
# Lưu ý: test vẫn không được sử dụng
train_validation_df = pd.concat([train_df, validation_df], ignore_index=True)

# In thông tin
print(f"Selected feature set: {best_validation['experiment']}")
print(f"Feature count: {len(feature_columns)}")
print(f"Final training rows: {len(train_validation_df)}")

Selected feature set: C_time_weather_lag
Feature count: 25
Final training rows: 14629


In [ ]:
# === TẢI SIÊU THAM SỐ BEST TỪ PHASE 6 ===
# Lấy hàng tốt nhất từ kết quả tìm kiếm (sắp xếp theo RMSE)
best_xgb = search_df.sort_values("RMSE").iloc[0]

# Tạo dictionary chứa tất cả siêu tham số cho XGBoost
xgb_parameters = {
    "n_estimators": int(best_xgb["n_estimators"]),  # Số cây
    "max_depth": int(best_xgb["max_depth"]),  # Độ sâu cây
    "learning_rate": float(best_xgb["learning_rate"]),  # Tốc độ học
    "min_child_weight": int(best_xgb["min_child_weight"]),  # Trọng số tối thiểu
    "objective": "reg:squarederror",  # Hàm mục tiêu
    "tree_method": "hist",  # Phương pháp xây dựng cây (CPU-friendly)
    "device": "cpu",  # Chạy trên CPU
    "n_jobs": -1,  # Sử dụng tất cả CPU cores
    "random_state": 42,  # Seed để tái lập kết quả
}

# In siêu tham số
print("Final XGBoost parameters:", xgb_parameters)

Final XGBoost parameters: {'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.03, 'min_child_weight': 3, 'objective': 'reg:squarederror', 'tree_method': 'hist', 'device': 'cpu', 'n_jobs': -1, 'random_state': 42}


## Fit final model on train + validation

In [ ]:
# === HUẤN LUYỆN MỌ HÌNH CUỐI CÙNG ===
# Tạo XGBoost model với các siêu tham số tốt nhất
final_model = XGBRegressor(**xgb_parameters)

# Bắt đầu đo thời gian
start_time = perf_counter()

# Huấn luyện model trên train + validation data
# (Sử dụng cả 2 tập để có nhiều dữ liệu huấn luyện hơn)
final_model.fit(train_validation_df[feature_columns], train_validation_df[target_column])

# Kết thúc đo thời gian
training_time = perf_counter() - start_time

# Dự đoán trên tập test (lần đầu tiên test được sử dụng)
test_predictions = final_model.predict(test_df[feature_columns])

# === TÍNH METRICS TRÊN TEST ===
# Tạo dictionary chứa metrics trên test
test_metrics = {
    "model": "XGBoost final",  # Tên model
    "feature_set": best_validation["experiment"],  # Feature set được sử dụng
    "split": "test",  # Dữ liệu test
    "MAE": mean_absolute_error(test_df[target_column], test_predictions),  # Sai số tuyệt đối trung bình
    "RMSE": mean_squared_error(test_df[target_column], test_predictions) ** 0.5,  # RMSE
    "R2": r2_score(test_df[target_column], test_predictions),  # R²
    "Training Time": training_time,  # Thời gian huấn luyện
}

# Chuyển thành DataFrame
final_metrics = pd.DataFrame([test_metrics])

# Hiển thị metrics
display(final_metrics.round(4))

# Lưu metrics vào file CSV
final_metrics.to_csv(metrics_dir / "best_model_final_metrics.csv", index=False)

# In thời gian huấn luyện
print(f"Training time: {training_time:.2f} seconds")

,model,feature_set,split,MAE,RMSE,R2,Training Time
0,XGBoost final,C_time_weather_lag,test,29.146,47.7517,0.9502,0.5351


Training time: 0.54 seconds


In [ ]:
# === TẠO BUNDLE MÔ HÌNH ===
# Tạo dictionary chứa tất cả thông tin cần để sử dụng mô hình sau này
# (model, features, và metadata)
model_bundle = {
    "model": final_model,  # Mô hình XGBoost đã huấn luyện
    "feature_columns": feature_columns,  # Danh sách tên features
    "target_column": target_column,  # Tên cột target
    "feature_set": best_validation["experiment"],  # Tên feature set
    "xgb_parameters": xgb_parameters,  # Siêu tham số XGBoost
}

# Định nghĩa đường dẫn lưu mô hình
model_path = models_dir / "best_model_xgboost.pkl"

# Lưu model bundle vào file pickle
with model_path.open("wb") as model_file:
    pickle.dump(model_bundle, model_file)

# === TÍNH ĐỘ QUAN TRỌNG FEATURES ===
# Tạo DataFrame chứa tên feature và độ quan trọng của nó
importance = pd.DataFrame({
    "feature": feature_columns,  # Tên features
    "importance": final_model.feature_importances_,  # Độ quan trọng (0-1)
}).sort_values("importance", ascending=False)  # Sắp xếp từ cao đến thấp

# Lưu độ quan trọng features vào file CSV
importance.to_csv(metrics_dir / "best_model_feature_importance.csv", index=False)

# In thông báo
print(f"Saved best model to: {model_path}")

Saved best model to: ..\models\best_model_xgboost.pkl


## Phase 11 conclusion

The selected XGBoost model is saved with its feature schema and parameters. PHASE 12 can load this bundle for the Streamlit prediction interface.